# Visual Feature Extraction — IEMOCAP

Same pipeline as MELD; no train/dev/test splits — single `.pt` per backbone.

Outputs: `Dataset/Processed/IEMOCAP/features/`
- `video_clip_temporal.pt`    — CLIP ViT-L/14  `(3, 768)`  per utterance  (begin / mid / end)
- `video_siglip2_temporal.pt` — SigLIP 2       `(3, 1152)` per utterance  (begin / mid / end)
- `video_openface_au.pt`      — OpenFace 3.0   `(8,)`      per utterance

Each temporal slot = mean of 3 frames sampled from that region of the clip.

IEMOCAP videos are speaker-cropped (single face, half-frame) — expect >95% face coverage.

In [7]:
import os, sys, cv2, tempfile, torch, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/IEMOCAP"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

FRAME_SAMPLE_FPS  = 2
BATCH_SIZE_FRAMES = 32
MAX_FRAMES        = 60
AU_CONF_THRESHOLD = 0.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Total utterances: {len(labels)}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060
Total utterances: 10039


In [8]:
def sample_frames(video_path: Path, fps=FRAME_SAMPLE_FPS, max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    src_fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(src_fps / fps)))
    frames, idx = [], 0
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % interval == 0:
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

## 1. CLIP ViT-L/14

In [ ]:
import clip

clip_model, clip_preprocess = clip.load("ViT-L/14", device=DEVICE)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad_(False)
CLIP_DIM = clip_model.visual.output_dim
N_FRAMES_PER_SEG = 3
print(f"CLIP dim: {CLIP_DIM}  |  frames/segment: {N_FRAMES_PER_SEG}")

def temporal_segments(frames, n=N_FRAMES_PER_SEG):
    """Split frames into (beginning, middle, end) lists of at most n frames each."""
    total = len(frames)
    if total == 0:
        return [], [], []
    mid    = total // 2
    half_n = n // 2
    beg = frames[:n]
    mid_seg = frames[max(0, mid - half_n): max(0, mid - half_n) + n]
    end = frames[max(0, total - n):]
    return beg, mid_seg, end

def extract_clip(frames):
    if not frames:
        return np.zeros((3, CLIP_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(frames):
        if not seg:
            vecs.append(np.zeros(CLIP_DIM, dtype=np.float32))
            continue
        pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in seg]
        tensors = torch.stack([clip_preprocess(img) for img in pil_frames]).to(DEVICE)
        with torch.no_grad():
            f = clip_model.encode_image(tensors).float()
            f = f / f.norm(dim=-1, keepdim=True)
        vecs.append(f.cpu().numpy().mean(axis=0))
    return np.stack(vecs)  # (3, CLIP_DIM)

out_path = FEATURES_DIR / "video_clip_temporal.pt"
if out_path.exists():
    print("Already exists — skipping")
else:
    features, skipped = {}, []
    for _, row in tqdm(labels.iterrows(), total=len(labels), desc="CLIP temporal"):
        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists():
            skipped.append(row["utt_id"]); continue
        frames = sample_frames(vid_path)
        if not frames:
            skipped.append(row["utt_id"]); continue
        features[row["utt_id"]] = extract_clip(frames)
    torch.save(features, out_path)
    print(f"Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)")
    if skipped: print(f"Skipped: {skipped}")

CLIP dim: 768  |  frames/segment: 3


CLIP temporal:   0%|          | 0/10039 [00:00<?, ?it/s]

## 2. SigLIP 2

In [ ]:
from transformers import AutoProcessor, AutoModel as HFAutoModel

siglip2_processor = AutoProcessor.from_pretrained("google/siglip2-so400m-patch14-384")
siglip2_model     = HFAutoModel.from_pretrained("google/siglip2-so400m-patch14-384")
siglip2_model.eval()
for p in siglip2_model.parameters():
    p.requires_grad_(False)
siglip2_model = siglip2_model.to(DEVICE)
SIGLIP2_DIM = siglip2_model.config.vision_config.hidden_size
print(f"SigLIP 2 dim: {SIGLIP2_DIM}")

def extract_siglip2(frames):
    if not frames:
        return np.zeros((3, SIGLIP2_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(frames):
        if not seg:
            vecs.append(np.zeros(SIGLIP2_DIM, dtype=np.float32))
            continue
        pil_frames = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in seg]
        inputs = siglip2_processor(images=pil_frames, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = siglip2_model.vision_model(**inputs)
        vecs.append(out.pooler_output.cpu().float().numpy().mean(axis=0))
    return np.stack(vecs)  # (3, SIGLIP2_DIM)

out_path = FEATURES_DIR / "video_siglip2_temporal.pt"
if out_path.exists():
    print("Already exists — skipping")
else:
    features, skipped = {}, []
    for _, row in tqdm(labels.iterrows(), total=len(labels), desc="SigLIP2 temporal"):
        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists():
            skipped.append(row["utt_id"]); continue
        frames = sample_frames(vid_path)
        if not frames:
            skipped.append(row["utt_id"]); continue
        features[row["utt_id"]] = extract_siglip2(frames)
    torch.save(features, out_path)
    print(f"Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)")
    if skipped: print(f"Skipped: {skipped}")

## 3. OpenFace 3.0 — Action Units

In [ ]:
OF3_WEIGHTS_DIR = Path.home() / ".cache" / "openface"
# OF3_WEIGHTS_DIR = Path("/path/to/OpenFace-3.0/weights")  # override if needed

from openface.face_detection import FaceDetector
from openface.multitask_model import MultitaskPredictor

of3_face = FaceDetector(
    model_path=str(OF3_WEIGHTS_DIR / "Alignment_RetinaFace.pth"),
    device=str(DEVICE), vis_threshold=AU_CONF_THRESHOLD,
)
of3_pred = MultitaskPredictor(
    model_path=str(OF3_WEIGHTS_DIR / "MTL_backbone.pth"),
    device=str(DEVICE),
)
print("OpenFace 3.0 ready")

def extract_au_of3(frames):
    au_list = []
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, frame in enumerate(frames):
            fp = os.path.join(tmpdir, f"f{i:04d}.jpg")
            cv2.imwrite(fp, frame)
            cropped_face, _ = of3_face.get_face(fp)
            if cropped_face is None:
                continue
            _, _, au_out = of3_pred.predict(cropped_face)
            au_list.append(au_out.cpu().float().numpy().flatten())
    if not au_list:
        return None
    return np.stack(au_list).mean(axis=0)

out_path = FEATURES_DIR / "video_openface_au.pt"
if out_path.exists():
    print("Already exists — skipping")
else:
    features:  dict[str, np.ndarray] = {}
    skipped:   list[str] = []
    zero_face: list[str] = []
    AU_DIM = None

    for _, row in tqdm(labels.iterrows(), total=len(labels), desc="AU OF3"):
        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists():
            skipped.append(row["utt_id"]); continue
        frames = sample_frames(vid_path)
        if not frames:
            skipped.append(row["utt_id"]); continue
        au_vec = extract_au_of3(frames)
        if au_vec is None:
            zero_face.append(row["utt_id"])
            if AU_DIM: features[row["utt_id"]] = np.zeros(AU_DIM, dtype=np.float32)
            continue
        if AU_DIM is None:
            AU_DIM = len(au_vec)
            print(f"AU dim: {AU_DIM}")
            for n in zero_face: features[n] = np.zeros(AU_DIM, dtype=np.float32)
        features[row["utt_id"]] = au_vec.astype(np.float32)

    if AU_DIM:
        for n in zero_face:
            if n not in features: features[n] = np.zeros(AU_DIM, dtype=np.float32)
    torch.save(features, out_path)
    cov = len(features) - len(zero_face)
    print(f"Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)")
    print(f"Face coverage: {cov}/{len(features)}  ({cov/max(len(features),1)*100:.1f}%)")
    if skipped:   print(f"Skipped: {skipped}")
    if zero_face: print(f"No face: {zero_face}")

Loading pretrained model from /home/jubaer/.cache/openface/Alignment_RetinaFace.pth
remove prefix 'module.'
Missing keys:0
Unused checkpoint keys:0
Used keys:300
Loading multitask model from /home/jubaer/.cache/openface/MTL_backbone.pth...


/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


OpenFace 3.0 ready


AU OF3:   0%|          | 0/10039 [00:00<?, ?it/s]

AU dim: 8
Saved: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP/features/video_openface_au.pt  (1.5 MB, 10039 entries)
Face coverage: 9960/10039  (99.2%)
No face: ['Ses02M_impro08_F014', 'Ses02M_impro08_F018', 'Ses05F_script02_1_M000', 'Ses05F_script02_1_M001', 'Ses05F_script02_1_M002', 'Ses05F_script02_1_M003', 'Ses05F_script02_1_M004', 'Ses05F_script02_1_M005', 'Ses05F_script02_1_M006', 'Ses05F_script02_1_M007', 'Ses05F_script02_1_M008', 'Ses05F_script02_1_M009', 'Ses05F_script02_1_M010', 'Ses05F_script02_1_M011', 'Ses05F_script02_1_M012', 'Ses05F_script02_1_M013', 'Ses05F_script02_1_M014', 'Ses05F_script02_1_M015', 'Ses05F_script02_1_M016', 'Ses05F_script02_1_M017', 'Ses05F_script02_1_M018', 'Ses05F_script02_1_M019', 'Ses05F_script02_1_M020', 'Ses05F_script02_1_M021', 'Ses05F_script02_1_M022', 'Ses05F_script02_1_M023', 'Ses05F_script02_1_M024', 'Ses05F_script02_1_M025', 'Ses05F_script02_1_M026', 'Ses05F_script02_1_M027', 'Ses05F_script02_1_M028', 'Ses05F_script02_1_M029'

In [ ]:
print("=== Verification ===")
checks = [
    ("video_clip_temporal",    (3, CLIP_DIM)),
    ("video_siglip2_temporal", (3, SIGLIP2_DIM)),
    ("video_openface_au",      (AU_DIM,)),
]
for tag, expected_shape in checks:
    pt = FEATURES_DIR / f"{tag}.pt"
    if not pt.exists(): print(f"  {tag}: MISSING"); continue
    d = torch.load(pt, weights_only=False)
    all_feats = np.stack(list(d.values()))
    shape = next(iter(d.values())).shape
    shape_ok = shape == expected_shape
    print(f"  {tag}: count={len(d)}/10039  shape={shape} {'✓' if shape_ok else '✗ expected '+str(expected_shape)}  "
          f"nan={np.isnan(all_feats).any()}  inf={np.isinf(all_feats).any()}")